# Six Star Finishers: Data Collection and Preparation

Collect public Hall of Fame records with `total_stars = 6`, validate them,
and prepare two related tables for a Power BI report.
The six marathons are Tokyo, Boston, London, Berlin, Chicago and New York City.

**Source:** [Abbott World Marathon Majors Hall of Fame](https://www.worldmarathonmajors.com/six-star/hall-of-fame).

**How to run:** Run the four code cells from top to bottom in Google Colab.
For a local Python environment, install `requests` and `pandas` first.
The CSV files are written to the current working directory.

The saved outputs are from the original successful Colab run on **6 October 2026**:
**27,964 finishers** and **167,784 marathon results**.
The source is live, so later downloads may return different counts.

## 1. Import libraries

Use `requests` for HTTP requests and `pandas` for table preparation and CSV export.
The download step also uses Python's built-in `datetime` module to record UTC time.

In [ ]:
import requests
import pandas as pd

## 2. Download all records

Use the public search endpoint called by Hall of Fame.
Request 1,000 records per batch, sorted by the source athlete ID.
`search_after` continues from the last record in each batch and allows pagination
beyond the default 10,000-result window.
Require an exact source count, check that it stays unchanged across requests,
and compare it with the downloaded count.

The endpoint is public and may change in the future. These requests do not lock
the live index into a fixed snapshot; count checks do not detect every possible
source update during a download.

In [ ]:
from datetime import datetime, timezone

api_url = "https://gxo3mfmtr4.execute-api.eu-west-1.amazonaws.com/halloffame/_search"

payload = {
    "size": 1000,
    "track_total_hits": True,
    "query": {"match": {"total_stars": "6"}},
    "sort": [{"athlete_id.keyword": "asc"}],
    "_source": [
        "athlete_id", "firstname", "lastname",
        "gender", "nationality", "total_stars", "results"
    ]
}

all_records = []
expected_total = None

while True:
    response = requests.post(api_url, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()

    total = data["hits"]["total"]

    if expected_total is None:
        expected_total = total["value"]

    if total["relation"] != "eq" or total["value"] != expected_total:
        raise ValueError("Source count is not exact or has changed.")

    batch = data["hits"]["hits"]

    if not batch:
        break

    all_records.extend(batch)
    payload["search_after"] = batch[-1]["sort"]

if len(all_records) != expected_total:
    raise ValueError("Incomplete download.")

downloaded_at_utc = datetime.now(timezone.utc).isoformat()

print(f"Downloaded records: {len(all_records):,}")
print(f"Source records: {expected_total:,}")

Downloaded records: 27,964
Source records: 27,964


## 3. Prepare tables

Create one row per finisher and one row per marathon result.
Preserve both `entry_id` and `athlete_id`; do not identify or merge people by name.
Resolve country and territory names using the source website's nationality dictionary.
Preserve the original finish-time text and add integer seconds for calculations.
Add the download completion timestamp in UTC to both tables.

`sequence` identifies the marathon: 1 = Tokyo, 2 = Boston, 3 = London,
4 = Berlin, 5 = Chicago and 6 = New York City. It is not the chronological
order of an athlete's races.

In [ ]:
people_data = []

for record in all_records:
    person = record["_source"].copy()
    person["entry_id"] = record["_id"]
    people_data.append(person)

finishers = pd.DataFrame(people_data).drop(columns="results")
finishers = finishers.rename(columns={
    "firstname": "first_name",
    "lastname": "last_name",
    "gender": "gender_code",
    "nationality": "nationality_code"
})
finishers["full_name"] = (
    finishers["first_name"] + " " + finishers["last_name"]
)

country_response = requests.get(
    "https://dfsqrpugige0x.cloudfront.net/prod/config_nationality.json",
    timeout=60
)
country_response.raise_for_status()
country_names = {
    item["value"]: item["label"]
    for item in country_response.json()
}
finishers["country_or_territory"] = (
    finishers["nationality_code"].map(country_names)
)

marathon_results = pd.json_normalize(
    people_data,
    record_path="results",
    meta=["entry_id", "athlete_id"]
)
marathon_results["marathon"] = marathon_results["sequence"].map({
    1: "Tokyo", 2: "Boston", 3: "London",
    4: "Berlin", 5: "Chicago", 6: "New York City"
})
marathon_results["finish_time_seconds"] = (
    pd.to_timedelta(marathon_results["finish_time"])
    .dt.total_seconds().astype("int64")
)

for table in (finishers, marathon_results):
    table["downloaded_at_utc"] = downloaded_at_utc

print(f"Finishers: {len(finishers):,}")
print(f"Marathon results: {len(marathon_results):,}")

Finishers: 27,964
Marathon results: 167,784


## 4. Validate and export

Check record counts, unique person and entry IDs, missing or blank values,
six distinct marathons per person, table links and finish-time conversions.
An `assert` stops execution if a check fails.
After all checks pass, export two CSV files with a semicolon separator,
UTF-8 with BOM and no DataFrame index column.

In [ ]:
assert len(finishers) == expected_total, "Source count mismatch."
assert finishers["entry_id"].is_unique, "Duplicate entry IDs."
assert finishers["athlete_id"].is_unique, "Duplicate athlete IDs."
assert finishers["total_stars"].eq(6).all(), "Unexpected star count."

for table in (finishers, marathon_results):
    assert not table.replace(
        r"^\s*$", pd.NA, regex=True
    ).isna().any().any(), "Missing or blank values found."

assert len(marathon_results) == len(finishers) * 6, "Result count mismatch."

result_counts = marathon_results.groupby("athlete_id")["marathon"].agg(
    ["size", "nunique"]
)
assert result_counts.eq(6).all().all(), "Expected six different marathons."

entry_ids = finishers.set_index("athlete_id")["entry_id"]
assert marathon_results["entry_id"].eq(
    marathon_results["athlete_id"].map(entry_ids)
).all(), "Invalid result links."

time_parts = marathon_results["finish_time"].str.split(
    ":", expand=True
).astype("int64")
expected_seconds = time_parts[0] * 3600 + time_parts[1] * 60 + time_parts[2]

assert marathon_results["finish_time_seconds"].eq(
    expected_seconds
).all(), "Incorrect time conversion."
assert marathon_results["finish_time_seconds"].gt(0).all(), "Nonpositive times."

finishers.to_csv(
    "finishers.csv", sep=";", encoding="utf-8-sig", index=False
)
marathon_results.to_csv(
    "marathon_results.csv", sep=";", encoding="utf-8-sig", index=False
)

print("All validation checks passed.")
print(f"finishers.csv: {len(finishers):,} rows")
print(f"marathon_results.csv: {len(marathon_results):,} rows")

All validation checks passed.
finishers.csv: 27,964 rows
marathon_results.csv: 167,784 rows


**Output files**

| File | One row represents | Main fields |
| --- | --- | --- |
| `finishers.csv` | One person | Source IDs, first and last name, full name, gender code, nationality code, country or territory, star count, download timestamp |
| `marathon_results.csv` | One person's result in one marathon | Source IDs, source event title, marathon, finish-time text, finish-time seconds, result year, sequence, download timestamp |

In Power BI, connect `finishers[athlete_id]` to `marathon_results[athlete_id]`
with a one-to-many relationship. Use `finish_time_seconds` for calculations.

**Data notes and limitations**

- Gender codes are kept unchanged: `M` = male, `F` = female; `N` is left uninterpreted.
- Nationality codes and country names follow the source dictionary. They are not automatically ISO codes or evidence of current residence.
- Finish times are durations in `h:mm:ss`; numeric durations are whole seconds. The source does not establish in these records whether a time is chip time or gun time.
- Hall of Fame includes wheelchair athletes. These exports do not have a verified competition-category field. All records are retained, and category is not inferred from finish time. Overall fastest times cannot be labelled as running records without identifying categories.
- Each person has one listed result per marathon. This is not their full race history.
- The smallest and largest result years, and their difference, describe the listed results. They do not establish when a person first completed the cycle or received a medal. No medal-date or cycle-duration column is inferred here.
- Completeness checks refer to the public `total_stars = 6` records returned by the endpoint, not to every person who has completed the six races outside this register.

**References**

- [Hall of Fame](https://www.worldmarathonmajors.com/six-star/hall-of-fame)
- [Elasticsearch pagination and search_after](https://www.elastic.co/docs/reference/elasticsearch/rest-apis/paginate-search-results)
- [AbbottWMM: Six Star finishers in the wheelchair division](https://www.worldmarathonmajors.com/content-hub/six-star-program-stands-on-the-brink-of-5-000)